In [1]:
import pickle
import pandas as pd
import deepchem as dc
import torch
from torch_geometric.data import Data

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\deepchem\feat\smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


In [2]:
data_train = pd.read_csv('data/train.csv')
data_train

,SMILES,eHOMO,eLUMO,DE_S0S1,f_S0S1,DE_S0T1,DES1T1
0,Cc1cccc(C)c1B1c2ccccc2B(c2c(C)cccc2C)c2c1cccc2...,-6.021883,-2.761141,2.6090,0.0008,2.5601,0.0489
1,C#Cc1ccc(OC)cc1OC,-5.912765,-0.808995,4.5937,0.2360,3.2077,1.3860
2,C#CC1COCCN1c1cc(Nc2ccc(C)cc2-c2cc(C#N)c(C#N)c(...,-5.459967,-2.838693,2.1244,0.0731,1.9606,0.1638
3,O=Cc1ccc(C(=O)c2ccc(C(=O)C(F)(F)F)cc2)cc1,-7.423270,-3.002506,3.3837,0.0021,2.7772,0.6065
4,Cn1c(-c2c(F)c(F)c(-c3ccc(-n4c5ccccc5c5cnccc54)...,-5.926900,-1.534200,3.8233,0.8733,3.0847,0.7386
...,...,...,...,...,...,...,...
48932,COc1ccc(-c2cc(-c3ccc(OC)cc3)[o+]c(-c3ccc(OC)cc...,-6.650500,-3.529900,2.6761,0.5841,2.1605,0.5156
48933,C[n+]1c2ccccc2cc2ccccc21,-7.427900,-3.930700,3.0142,0.0670,1.9792,1.0350
48934,C[n+]1c2ccccc2c(-c2ccccc2)c2ccccc21,-7.321800,-3.899400,2.9292,0.1330,1.9478,0.9814
48935,Cc1cc(C)c(-c2c3ccccc3[n+](C)c3ccccc23)c(C)c1,-7.138100,-3.895600,2.4928,0.0000,1.9542,0.5386


In [3]:
data_val = pd.read_csv('data/val.csv')
data_val

,SMILES,eHOMO,eLUMO,DE_S0S1,f_S0S1,DE_S0T1,DES1T1
0,C(=Cn1c2ccccc2c2ccccc21)c1cc2cnccc2[nH]1,-5.459200,-1.514600,3.5062,1.0614,2.4254,1.0808
1,CN(C)c1ccc(-c2ccc(-c3ccc4c(n3)[SiH2]c3ccccc3-4...,-5.365800,-1.612300,3.3071,0.7337,2.5139,0.7932
2,Cn1cnc2cc(-c3ccc4c(c3)sc3cc(-n5c6ccccc6c6ccccc...,-5.648300,-1.539600,3.5836,0.6704,2.8872,0.6964
3,CN(C)c1ccc(-c2ccc(-c3nc4ccccc4s3)n2C)c2ccccc12,-5.426500,-1.554000,3.3721,0.7327,2.5025,0.8696
4,N#CC(C#N)=Cc1ccc(-c2ccc(C(=O)C(F)(F)F)cc2)s1,-6.600397,-3.317342,3.0235,1.1570,1.8567,1.1668
...,...,...,...,...,...,...,...
2638,CN(C)c1ccc(-c2ccc3c(c2)oc2ccccc23)c2ccccc12,-5.488539,-1.461252,3.5178,0.4513,2.5427,0.9751
2639,Clc1ccc2c(c1)c1cc(Cl)ccc1n2-c1ccc2c(c1)oc1cc(-...,-5.714700,-1.853600,3.3915,0.9471,2.6392,0.7523
2640,O=S1(=O)c2cc(Sn3c4ccncc4c4cnccc43)ccc2-c2ccc(S...,-6.373500,-2.150000,3.6650,0.3864,2.7175,0.9475
2641,C=Cc1cccc2c(=O)c3cccc(-c4ccc5c(c4)c(=O)ccn5-c4...,-5.766096,-1.904526,3.3600,0.2020,2.6418,0.7182


In [4]:
data_test = pd.read_csv('data/test.csv')
data_test

,SMILES,eHOMO,eLUMO,DE_S0S1,f_S0S1,DE_S0T1,DES1T1
0,CN(C)c1ccc2c(n1)[SiH2]c1ccccc1-2,-5.326400,-1.269100,3.4513,0.0232,2.6968,0.7545
1,N#CC(C#N)=Cc1ccc([SH]2C=NC(c3cccc(C=O)c3)=C2)s1,-6.204471,-2.681683,2.8004,0.0433,2.0610,0.7394
2,O=Cc1ccc(C(=O)N(c2ccc(F)cc2)c2ccc(F)cc2)cc1-c1...,-6.160661,-2.351337,3.1661,0.0842,2.6475,0.5186
3,c1ccc2sc(-c3ccc(-n4c5ccccc5c5ccccc54)o3)nc2c1,-5.603400,-1.809000,3.3251,0.8407,2.4994,0.8257
4,Cc1cccc2c3cccc(C)c3n(-c3ccc4c(c3)sc3cc(-c5ccc6...,-5.615900,-2.161400,3.1448,0.0002,2.7505,0.3943
...,...,...,...,...,...,...,...
2639,O=c1cc[nH]c2ccc(-c3cccc4c(=O)ccn(S)c34)cc12,-6.024876,-1.691461,3.7974,0.1481,2.9632,0.8342
2640,CN(CN(c1ccccc1)c1ccccc1)c1ccccc1-c1ccc2cccc3c2...,-5.479832,-2.746991,2.2019,0.0005,1.9006,0.3013
2641,N#CC(C#N)=C1c2ccccc2C(=O)c2cc(-c3ccc4c(sc5cccc...,-6.211818,-3.422922,2.4359,0.0176,2.1654,0.2705
2642,Cn1cnc2ccc(-c3c(-c4ccc5c(c4)sc4ccccc45)csc3-c3...,-5.714938,-2.908082,2.2763,0.0467,1.9126,0.3637


In [5]:
torch.manual_seed(42)

featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
smiles_train = data_train["SMILES"]
X_train = featurizer.featurize(smiles_train)
smiles_val = data_val["SMILES"]
X_val = featurizer.featurize(smiles_val)
smiles_test = data_test["SMILES"]
X_test = featurizer.featurize(smiles_test)

In [6]:
torch.manual_seed(42)

target_train_eHOMO = data_train["eHOMO"]
target_val_eHOMO = data_val["eHOMO"]
target_test_eHOMO = data_test["eHOMO"]

target_train_eHOMO = torch.tensor(target_train_eHOMO.values, dtype=torch.float32)
target_val_eHOMO = torch.tensor(target_val_eHOMO.values, dtype=torch.float32)
target_test_eHOMO = torch.tensor(target_test_eHOMO.values, dtype=torch.float32)

X_train_list = []
for graph_data in X_train:
    node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
    edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
    edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
    data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
    X_train_list.append(data)
    
X_val_list = []
for graph_data in X_val:
    node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
    edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
    edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
    data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
    X_val_list.append(data)

X_test_list = []
for graph_data in X_test:
    node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
    edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
    edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
    data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
    X_test_list.append(data)

In [7]:
torch.manual_seed(42)

with open('data/data_train_list.pkl', 'wb') as f:
    pickle.dump(X_train_list, f)
with open('data/data_val_list.pkl', 'wb') as f:
    pickle.dump(X_val_list, f)
with open('data/data_test_list.pkl', 'wb') as f:
    pickle.dump(X_test_list, f)
with open('data/target_train_eHOMO.pkl', 'wb') as f:
    pickle.dump(target_train_eHOMO, f)
with open('data/target_val_eHOMO.pkl', 'wb') as f:
    pickle.dump(target_val_eHOMO, f)
with open('data/target_test_eHOMO.pkl', 'wb') as f:
    pickle.dump(target_test_eHOMO, f)

In [8]:
torch.manual_seed(42)

target_train_eLUMO = data_train["eLUMO"]
target_val_eLUMO = data_val["eLUMO"]
target_test_eLUMO = data_test["eLUMO"]

target_train_eLUMO = torch.tensor(target_train_eLUMO.values, dtype=torch.float32)
target_val_eLUMO = torch.tensor(target_val_eLUMO.values, dtype=torch.float32)
target_test_eLUMO = torch.tensor(target_test_eLUMO.values, dtype=torch.float32)

with open('data/target_train_eLUMO.pkl', 'wb') as f:
    pickle.dump(target_train_eLUMO, f)
with open('data/target_val_eLUMO.pkl', 'wb') as f:
    pickle.dump(target_val_eLUMO, f)
with open('data/target_test_eLUMO.pkl', 'wb') as f:
    pickle.dump(target_test_eLUMO, f)

In [9]:
torch.manual_seed(42)

target_train_DE_S0S1 = data_train["DE_S0S1"]
target_val_DE_S0S1 = data_val["DE_S0S1"]
target_test_DE_S0S1 = data_test["DE_S0S1"]

target_train_DE_S0S1 = torch.tensor(target_train_DE_S0S1.values, dtype=torch.float32)
target_val_DE_S0S1 = torch.tensor(target_val_DE_S0S1.values, dtype=torch.float32)
target_test_DE_S0S1 = torch.tensor(target_test_DE_S0S1.values, dtype=torch.float32)

with open('data/target_train_DE_S0S1.pkl', 'wb') as f:
    pickle.dump(target_train_DE_S0S1, f)
with open('data/target_val_DE_S0S1.pkl', 'wb') as f:
    pickle.dump(target_val_DE_S0S1, f)
with open('data/target_test_DE_S0S1.pkl', 'wb') as f:
    pickle.dump(target_test_DE_S0S1, f)

In [10]:
torch.manual_seed(42)

target_train_f_S0S1 = data_train["f_S0S1"]
target_val_f_S0S1 = data_val["f_S0S1"]
target_test_f_S0S1 = data_test["f_S0S1"]

target_train_f_S0S1 = torch.tensor(target_train_f_S0S1.values, dtype=torch.float32)
target_val_f_S0S1 = torch.tensor(target_val_f_S0S1.values, dtype=torch.float32)
target_test_f_S0S1 = torch.tensor(target_test_f_S0S1.values, dtype=torch.float32)

with open('data/target_train_f_S0S1.pkl', 'wb') as f:
    pickle.dump(target_train_f_S0S1, f)
with open('data/target_val_f_S0S1.pkl', 'wb') as f:
    pickle.dump(target_val_f_S0S1, f)
with open('data/target_test_f_S0S1.pkl', 'wb') as f:
    pickle.dump(target_test_f_S0S1, f)

In [11]:
torch.manual_seed(42)

target_train_DE_S0T1 = data_train["DE_S0T1"]
target_val_DE_S0T1 = data_val["DE_S0T1"]
target_test_DE_S0T1 = data_test["DE_S0T1"]

target_train_DE_S0T1 = torch.tensor(target_train_DE_S0T1.values, dtype=torch.float32)
target_val_DE_S0T1 = torch.tensor(target_val_DE_S0T1.values, dtype=torch.float32)
target_test_DE_S0T1 = torch.tensor(target_test_DE_S0T1.values, dtype=torch.float32)

with open('data/target_train_DE_S0T1.pkl', 'wb') as f:
    pickle.dump(target_train_DE_S0T1, f)
with open('data/target_val_DE_S0T1.pkl', 'wb') as f:
    pickle.dump(target_val_DE_S0T1, f)
with open('data/target_test_DE_S0T1.pkl', 'wb') as f:
    pickle.dump(target_test_DE_S0T1, f)

In [12]:
torch.manual_seed(42)

target_train_DES1T1 = data_train["DES1T1"]
target_val_DES1T1 = data_val["DES1T1"]
target_test_DES1T1 = data_test["DES1T1"]

target_train_DES1T1 = torch.tensor(target_train_DES1T1.values, dtype=torch.float32)
target_val_DES1T1 = torch.tensor(target_val_DES1T1.values, dtype=torch.float32)
target_test_DES1T1 = torch.tensor(target_test_DES1T1.values, dtype=torch.float32)

with open('data/target_train_DES1T1.pkl', 'wb') as f:
    pickle.dump(target_train_DES1T1, f)
with open('data/target_val_DES1T1.pkl', 'wb') as f:
    pickle.dump(target_val_DES1T1, f)
with open('data/target_test_DES1T1.pkl', 'wb') as f:
    pickle.dump(target_test_DES1T1, f)